# Resolve implementations with the backend factory

[All How-to Guides](README.md)

The backend factory maps a component class and a `Dataset` backend to a concrete implementation. Inspect the built-in FAISS registrations, then register a custom component for pandas and Spark.

For local Spark, set `JAVA_HOME` to Java 17.

## Prepare the same data on both backends

Create a pandas-backed `Dataset` with three revenue values and move it to Spark with `to_backend()`. If `PYSPARK_PYTHON` is unset, use the current kernel's interpreter for Spark workers.

In [1]:
import os
import sys

from pyspark.sql import SparkSession, functions as F

from hypex.dataset import Dataset, TargetRole
from hypex.dataset.backends import PandasDataset, SparkDataset
from hypex.extensions.faiss import FaissExtension
from hypex.utils import BackendsEnum
from hypex.utils.registry import BackendFactory, backend_factory

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx backend factory")
    .getOrCreate()
)
local = Dataset(
    roles={"revenue": TargetRole(float)},
    data={"data": {"revenue": [2.0, 4.0, 9.0]}},
)
distributed = local.to_backend(BackendsEnum.spark, session=spark)
(local.backend_type.value, distributed.backend_type.value)

('pandas', 'spark')

The same metric is now available on `pandas` and `spark`.

## Resolve a built-in implementation

`backend_factory.registry` returns a snapshot of current registrations. Inspect the entries for `FaissExtension` and call `resolve_backend()` on both inputs. The method returns the implementation class that HypEx can instantiate.

In [2]:
registrations = backend_factory.registry[FaissExtension]
print("Registered:")
for backend, implementation in registrations.items():
    print(f"{backend.__name__}: {implementation.__name__}")

print("Resolved:")
for data in (local, distributed):
    implementation = backend_factory.resolve_backend(FaissExtension, data)
    print(f"{data.backend_type.value}: {implementation.__name__}")

Registered:
PandasDataset: PandasFaissExtension
SparkDataset: SparkFaissExtension
Resolved:
pandas: PandasFaissExtension
spark: SparkFaissExtension


The `FaissExtension` key resolves to `PandasFaissExtension` or `SparkFaissExtension` depending on the input `Dataset`. HypEx experiments use the factory internally.

## Register a custom component

Create a separate `BackendFactory()` to keep the example's registrations isolated from the built-ins. Register the pandas implementation with a decorator and register the already-defined Spark implementation with `register_explicit()`.

Both implementations provide `compute()`. The shared `RevenueMean.calc()` method resolves the class and reports missing registrations. The Spark implementation aggregates on Spark and returns one number to the driver.

In [3]:
factory = BackendFactory()


class RevenueMean:
    @classmethod
    def calc(cls, data):
        implementation = factory.resolve_backend(cls, data)
        if implementation is None:
            raise NotImplementedError(f"RevenueMean does not support {data.backend_type.value}")
        return implementation.compute(data)


@factory.register(RevenueMean, PandasDataset)
class PandasRevenueMean:
    @staticmethod
    def compute(data):
        return float(data.select("revenue").mean())


class SparkRevenueMean:
    @staticmethod
    def compute(data):
        return float(data.data.agg(F.avg("revenue")).first()[0])


factory.register_explicit(RevenueMean, SparkDataset, SparkRevenueMean)
{data.backend_type.value: RevenueMean.calc(data) for data in (local, distributed)}

{'pandas': 5.0, 'spark': 5.0}

Both calls returned 5.0. Each backend aggregates the data through the same component interface.

## Handle a missing registration

Remove the Spark implementation from the local factory. `resolve_backend()` returns `None` and the custom component reports that it cannot handle the Spark input. Stop the local session afterward.

In [4]:
factory.unregister(RevenueMean, SparkDataset)
print("Resolved:", factory.resolve_backend(RevenueMean, distributed))

try:
    RevenueMean.calc(distributed)
except NotImplementedError as error:
    print(error)
finally:
    spark.stop()

Resolved: None
RevenueMean does not support spark


The error identifies the backend that needs a registration. In a package, register the implementation when its module is imported so it is available on the first call. See the [backend registry API](../../../hypex/utils/registry.py) for the full interface.